# Keyword Index EDA

Read-only exploration for Feature 06 keyword indexing. Run `uv run -m uni_rag_agent index keyword` after extraction and data-summary chunks exist, then use this notebook to inspect `chunk_fts` coverage, stale rows, source-type distribution, and small keyword smoke queries.

Safety boundary: this notebook reads generated app data only. It opens SQLite in read-only mode, enables `PRAGMA query_only = ON`, and must not mutate SQLite, indexes, `Courses/`, or source course files. Clear outputs and execution counts before committing.

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "context"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "Could not find repo root. Start Jupyter from this project or notebooks/."
    )


repo_root = find_repo_root()
sqlite_path = repo_root / "data" / "uni_rag.sqlite"
if not sqlite_path.is_file():
    raise FileNotFoundError(
        f"SQLite database not found: {sqlite_path}. Run storage, inventory, extraction, and keyword indexing first."
    )

connection = sqlite3.connect(f"{sqlite_path.resolve().as_uri()}?mode=ro", uri=True)
connection.execute("PRAGMA query_only = ON")

In [ ]:
eligible_chunks = pd.read_sql_query(
    """
    SELECT
        chunks.id AS chunk_id,
        chunks.source_type,
        chunks.title,
        files.path AS file_path,
        courses.name AS course_name
    FROM chunks
    JOIN files ON files.id = chunks.file_id
    LEFT JOIN courses ON courses.id = files.course_id
    WHERE files.index_status = 'indexed'
      AND TRIM(chunks.text) <> ''
      AND chunks.source_type IN ('document', 'slides', 'notebook', 'code', 'data_schema', 'transcript')
    """,
    connection,
)

fts_rows = pd.read_sql_query(
    """
    SELECT chunk_id, title, course_name, file_path, source_type
    FROM chunk_fts
    """,
    connection,
)

eligible_by_source = (
    eligible_chunks.groupby("source_type", dropna=False)
    .size()
    .rename("eligible_chunks")
)
fts_by_source = fts_rows.groupby("source_type", dropna=False).size().rename("fts_rows")
source_coverage = (
    pd.concat([eligible_by_source, fts_by_source], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
    .sort_values("source_type")
)

pd.DataFrame(
    [
        {"metric": "eligible_chunks", "count": len(eligible_chunks)},
        {"metric": "fts_rows", "count": len(fts_rows)},
    ]
)

In [ ]:
missing_fts_rows = pd.read_sql_query(
    """
    SELECT eligible.chunk_id, eligible.source_type, eligible.file_path, eligible.course_name
    FROM (
        SELECT
            chunks.id AS chunk_id,
            chunks.source_type,
            files.path AS file_path,
            courses.name AS course_name
        FROM chunks
        JOIN files ON files.id = chunks.file_id
        LEFT JOIN courses ON courses.id = files.course_id
        WHERE files.index_status = 'indexed'
          AND TRIM(chunks.text) <> ''
          AND chunks.source_type IN ('document', 'slides', 'notebook', 'code', 'data_schema', 'transcript')
    ) AS eligible
    LEFT JOIN chunk_fts ON chunk_fts.chunk_id = eligible.chunk_id
    WHERE chunk_fts.chunk_id IS NULL
    ORDER BY eligible.chunk_id
    """,
    connection,
)

stale_fts_rows = pd.read_sql_query(
    """
    SELECT chunk_fts.chunk_id, chunk_fts.source_type, chunk_fts.file_path
    FROM chunk_fts
    LEFT JOIN chunks ON chunks.id = chunk_fts.chunk_id
    LEFT JOIN files ON files.id = chunks.file_id
    WHERE chunks.id IS NULL OR files.index_status <> 'indexed'
    ORDER BY chunk_fts.chunk_id
    """,
    connection,
)

field_coverage = pd.DataFrame(
    [
        {
            "field": "title",
            "non_empty_rows": int(
                fts_rows["title"].fillna("").str.strip().astype(bool).sum()
            )
            if not fts_rows.empty
            else 0,
        },
        {
            "field": "course_name",
            "non_empty_rows": int(
                fts_rows["course_name"].fillna("").str.strip().astype(bool).sum()
            )
            if not fts_rows.empty
            else 0,
        },
        {
            "field": "file_path",
            "non_empty_rows": int(
                fts_rows["file_path"].fillna("").str.strip().astype(bool).sum()
            )
            if not fts_rows.empty
            else 0,
        },
    ]
)

coverage_summary = pd.DataFrame(
    [
        {"metric": "missing_fts_rows", "count": len(missing_fts_rows)},
        {"metric": "stale_fts_rows", "count": len(stale_fts_rows)},
    ]
)
coverage_summary

In [ ]:
smoke_terms = ["bm25", "mapreduce", "search"]
smoke_rows = []
for term in smoke_terms:
    fts_query = f'"{term}"'
    try:
        result = pd.read_sql_query(
            """
            SELECT COUNT(*) AS result_count
            FROM chunk_fts
            JOIN chunks ON chunks.id = chunk_fts.chunk_id
            JOIN files ON files.id = chunks.file_id
            WHERE chunk_fts MATCH ?
              AND files.index_status = 'indexed'
            """,
            connection,
            params=(fts_query,),
        )
        smoke_rows.append(
            {
                "term": term,
                "result_count": int(result.loc[0, "result_count"]),
                "error": None,
            }
        )
    except Exception as exc:
        smoke_rows.append({"term": term, "result_count": 0, "error": str(exc)})

smoke_query_counts = pd.DataFrame(smoke_rows)
smoke_query_counts

In [ ]:
if not source_coverage.empty:
    ax = source_coverage.set_index("source_type")[["eligible_chunks", "fts_rows"]].plot(
        kind="bar"
    )
    ax.set_title("Keyword index coverage by source type")
    ax.set_xlabel("source_type")
    ax.set_ylabel("rows")
    plt.tight_layout()
else:
    print("No keyword source coverage rows to plot.")

In [ ]:
ax = coverage_summary.set_index("metric")["count"].plot(kind="bar")
ax.set_title("Missing and stale keyword FTS rows")
ax.set_xlabel("metric")
ax.set_ylabel("rows")
plt.tight_layout()

In [ ]:
if not smoke_query_counts.empty:
    ax = smoke_query_counts.set_index("term")["result_count"].plot(kind="bar")
    ax.set_title("Keyword smoke-query result counts")
    ax.set_xlabel("term")
    ax.set_ylabel("results")
    plt.tight_layout()
else:
    print("No keyword smoke-query rows to plot.")